---
title: "Predictions: Ensemble Workflow"
subtitle: "One Poisson model per team, round by round"
author: "Miguel R."
date: today
---

The **ensemble workflow** starts from the exact same per-team Poisson models as the
[naive workflow](rounds_naive.ipynb), then adds a **player layer** on top.

For each team we compute the average Elo of its four best attackers and compare it
against the average Elo of the opponent's defensive block (four best defenders, best
midfielder and best goalkeeper). The ratio of the two, raised to a damping
exponent (`(atk/def) ** 2.5`, tuned on the completed rounds - see
[model vs reality](compare_results.ipynb)), multiplies the predicted goals: a team whose attack outclasses the opposing
defense gets a boost, and a team facing a stronger defensive block gets discounted.

The point of running both workflows on identical fixtures is to answer one question:
**does player-level information actually improve the prediction, or is the team-level
history enough?**

## 1. Setup

All the heavy lifting lives in the project modules:

- `fixtures.py` holds the tournament calendar (this is the only file that changes between rounds),
- `model_utils.py` trains the per-team Poisson models,
- `report_utils.py` generates, stores and scores the predictions.


In [1]:
from model_utils import Workflow
from report_utils import predict_all_rounds, round_detail, scoreboard

## 2. Generating the predictions

One call predicts every round in the calendar. For each round, the model trains only on
matches played *before* the round started, so these predictions are fully reproducible:
re-running this notebook after the tournament produces the same numbers that were
published in real time.

Each round is written to `predictions/ensemble_<round>.csv`, separate from the other
workflow's files, so the two approaches can always be compared side by side.


In [2]:
predictions = predict_all_rounds(Workflow.ATK_DEF, "ensemble")

## 3. Round by round

For every completed round we show the predicted score next to the real one.
`correct` marks whether the predicted winner matched the actual result
(a real draw counts as a miss, since the model virtually never predicts one).


### 3.1 Group stage - Matchday 1

Twenty-four matches, and the model's first contact with reality. Group-stage matches allow draws, but a Poisson model almost never predicts identical expected goals for both sides, so every real draw counts against us here.

In [3]:
round_detail("ensemble", "first_round")

,team_1,team_2,predicted_score,real_score,winner_pred,winner_real,correct
0,Mexico,South Africa,3.3 - 0.23,2 - 0,Mexico,Mexico,True
1,South Korea,Czechia,1.8 - 1.31,2 - 1,South Korea,South Korea,True
2,Canada,Bosnia and Herzegovina,1.53 - 1.06,1 - 1,Canada,Draw,False
3,Qatar,Switzerland,0.21 - 7.41,1 - 1,Switzerland,Draw,False
4,Brazil,Morocco,1.99 - 0.98,1 - 1,Brazil,Draw,False
5,Haiti,Scotland,1.47 - 6.47,0 - 1,Scotland,Scotland,True
6,USA,Paraguay,4.09 - 0.75,4 - 1,USA,USA,True
7,Australia,Türkiye,0.54 - 2.66,2 - 0,Türkiye,Australia,False
8,Germany,Curaçao,19.67 - 0.2,7 - 1,Germany,Germany,True
9,Côte d'Ivoire,Ecuador,2.43 - 1.66,1 - 0,Côte d'Ivoire,Côte d'Ivoire,True


### 3.2 Group stage - Matchday 2

By matchday 2 the favorites usually start separating from the rest of the group, which tends to make matches easier to call.

In [4]:
round_detail("ensemble", "second_round")

,team_1,team_2,predicted_score,real_score,winner_pred,winner_real,correct
0,Czechia,South Africa,1.52 - 0.01,1 - 1,Czechia,Draw,False
1,Mexico,South Korea,1.35 - 2.0,1 - 0,South Korea,Mexico,False
2,Switzerland,Bosnia and Herzegovina,1.56 - 1.48,4 - 1,Switzerland,Switzerland,True
3,Canada,Qatar,1.25 - 0.92,6 - 0,Canada,Canada,True
4,Brazil,Haiti,3.32 - 0.0,3 - 0,Brazil,Brazil,True
5,Scotland,Morocco,0.38 - 2.43,0 - 1,Morocco,Morocco,True
6,Türkiye,Paraguay,2.35 - 1.23,0 - 1,Türkiye,Paraguay,False
7,USA,Australia,2.76 - 0.31,2 - 0,USA,USA,True
8,Germany,Côte d'Ivoire,2.15 - 0.0,2 - 1,Germany,Germany,True
9,Ecuador,Curaçao,3.73 - 0.97,0 - 0,Ecuador,Draw,False


### 3.3 Group stage - Matchday 3

The hardest matchday to predict: teams that already qualified rotate their squads, and teams that are already out play with nothing to lose. None of that appears in the training data.

In [5]:
round_detail("ensemble", "third_round")

,team_1,team_2,predicted_score,real_score,winner_pred,winner_real,correct
0,Czechia,Mexico,0.6 - 2.61,0 - 3,Mexico,Mexico,True
1,South Africa,South Korea,0.02 - 4.1,1 - 0,South Korea,South Africa,False
2,Switzerland,Canada,0.0 - 1.83,2 - 1,Canada,Switzerland,False
3,Bosnia and Herzegovina,Qatar,1.13 - 0.3,3 - 1,Bosnia and Herzegovina,Bosnia and Herzegovina,True
4,Scotland,Brazil,0.63 - 1.83,0 - 3,Brazil,Brazil,True
5,Morocco,Haiti,2.54 - 0.0,4 - 2,Morocco,Morocco,True
6,Türkiye,USA,0.01 - 2.18,3 - 2,USA,Türkiye,False
7,Paraguay,Australia,1.27 - 0.16,0 - 0,Paraguay,Draw,False
8,Curaçao,Côte d'Ivoire,0.28 - 1.26,0 - 2,Côte d'Ivoire,Côte d'Ivoire,True
9,Ecuador,Germany,0.17 - 7.33,2 - 1,Germany,Ecuador,False


### 3.4 Round of 32

First knockout round. A caveat on scoring: the match history records the score after 90 or 120 minutes, so a match decided on penalties is recorded as a draw and counts against the model even when its pick advanced.

In [6]:
round_detail("ensemble", "round_of_32")

,team_1,team_2,predicted_score,real_score,winner_pred,winner_real,correct
0,South Africa,Canada,0.19 - 2.79,0 - 1,Canada,Canada,True
1,Brazil,Japan,2.81 - 0.95,2 - 1,Brazil,Brazil,True
2,Germany,Paraguay,3.77 - 0.64,1 - 1,Germany,Draw,False
3,Netherlands,Morocco,4.88 - 2.7,1 - 1,Netherlands,Draw,False
4,Côte d'Ivoire,Norway,0.68 - 3.07,1 - 2,Norway,Norway,True
5,France,Sweden,3.19 - 0.38,3 - 0,France,France,True
6,Mexico,Ecuador,0.85 - 0.42,2 - 0,Mexico,Mexico,True
7,England,Congo DR,12.74 - 1.8,2 - 1,England,England,True
8,Belgium,Senegal,0.76 - 1.75,3 - 2,Senegal,Belgium,False
9,USA,Bosnia and Herzegovina,10.14 - 0.92,2 - 0,USA,USA,True


### 3.5 Round of 16

Second knockout round — the same penalty-shootout caveat as Round of 32 applies here too.

In [7]:
round_detail("ensemble", "round_of_16")

,team_1,team_2,predicted_score,real_score,winner_pred,winner_real,correct
0,Morocco,Canada,3.52 - 0.44,3 - 0,Morocco,Morocco,True
1,France,Paraguay,3.7 - 1.5,1 - 0,France,France,True
2,Brazil,Norway,2.54 - 2.33,1 - 2,Brazil,Norway,False
3,Mexico,England,1.23 - 2.2,2 - 3,England,England,True
4,Spain,Portugal,1.71 - 0.63,1 - 0,Spain,Spain,True
5,USA,Belgium,1.36 - 1.6,1 - 4,Belgium,Belgium,True
6,Argentina,Egypt,3.39 - 0.18,3 - 2,Argentina,Argentina,True
7,Colombia,Switzerland,0.44 - 1.92,0 - 0,Switzerland,Draw,False


### 3.6 Quarter-finals

Predictions for the current round. The evaluation below fills in automatically as matches are played and the ratings are re-scraped.

In [8]:
round_detail("ensemble", "quarter_finals")

,team_1,team_2,predicted_score,predicted_winner
0,Morocco,France,0.96 - 3.55,France
1,Spain,Belgium,2.12 - 0.63,Spain
2,Norway,England,2.51 - 2.88,England
3,Argentina,Switzerland,1.63 - 0.82,Argentina


## 4. Scoreboard

Winner accuracy and mean absolute goal error per round, over the matches played so far.


In [9]:
scoreboard(prefixes=("ensemble",))

,round,matches,ensemble_accuracy,ensemble_goal_mae
0,Group stage - Matchday 1,24,0.542,2.59
1,Group stage - Matchday 2,24,0.625,1.75
2,Group stage - Matchday 3,24,0.542,1.86
3,Round of 32,16,0.688,1.68
4,Round of 16,8,0.750,1.08


## 5. Discussion

Two patterns are worth calling out.

First, the deeper the tournament goes, the closer the expected goals of the two sides
get. That is not a bug: the knockout bracket filters out weak teams, so the survivors
are increasingly similar in strength, and the model's honest answer is "this one is
close to a coin flip". Many of these matches will realistically be decided in extra
time or on penalties, which no goals-based model can call.

Second, winner accuracy and goal error move in opposite directions as the stakes rise.
Calling the winner gets harder (closer matchups), while the *scores* the model misses
by grow because knockout football produces more extreme results than the friendlies
and qualifiers it trained on.
